In [1]:
from langchain_openai import ChatOpenAI
import os 
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

def read_email_tool(email_id  : str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID : {email_id}"

def send_email_tool(recipient : str, subject: str, body: str) -> str :
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

model = ChatOpenAI(
    model = "nvidia/nemotron-3-super-120b-a12b:free",
    api_key = os.environ["OPENROUTER_API_KEY"],
    base_url = "https://openrouter.ai/api/v1"
)
agent = create_agent(
    model = model,
    tools = [read_email_tool, send_email_tool],
    checkpointer= InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool": {
                    "allowed_decisions" : ["approve", "edit", "reject"]
                },
                "read_email_tool" : False,
            }
        )
    ]
)



In [2]:
config = {"configurable" : {"thread_id" : "test-edit"}}

#step-1 : Request
result = agent.invoke(
    {"messages" : [HumanMessage(content = "Send email to wrong@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config
)

In [3]:
result

{'messages': [HumanMessage(content="Send email to wrong@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='907523c3-4d4a-492a-960a-5cef59d8ab71'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 404, 'total_tokens': 494, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 36, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'system_fingerprint': None, 'id': 'gen-1782956893-M3dCx85bH4bssc3i1lKF', 'finish_reason': 'tool_calls', 'logpro

In [4]:
## Step-2 edit and Approve
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! Editing")

    result = agent.invoke(
        Command(
            resume = {
                "decisions":[
                    {
                        "type" : "edit",
                        "edited_action" : {
                            "name" : "send_email_tool",
                            "args" : {
                                "recipient" : "Correct@email.com",
                                "subject" : "Corrected Subject",
                                "body" : "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

Paused! Editing


In [5]:
print(f" Result : {result["messages"][-1].content}")

 Result : 


In [6]:
result

{'messages': [HumanMessage(content="Send email to wrong@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='907523c3-4d4a-492a-960a-5cef59d8ab71'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 404, 'total_tokens': 494, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 36, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'system_fingerprint': None, 'id': 'gen-1782956893-M3dCx85bH4bssc3i1lKF', 'finish_reason': 'tool_calls', 'logpro

In [8]:
from langgraph.types import Command

while "__interrupt__" in result:
    print("Paused! Reviewing...")
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config
    )

print(result["messages"][-1].content)

The email has been successfully sent to **wrong@test.com** with the subject **"Hello"** and body **"How are you?"**.
